# Lesson 7:

Docker is a platform for building, shipping, and running applications inside **containers** — lightweight, portable units that package your code together with everything it needs to run (dependencies, runtime, config).

## Why Docker?

| Problem | Docker Solution |
|---|---|
| "Works on my machine" | Same container runs everywhere |
| Dependency conflicts | Each container has isolated dependencies |
| Complex setup | One `docker run` command |
| Scaling apps | Spin up multiple containers easily |

## Core Concepts

- **Image** — A read-only blueprint (like a class). Built from a `Dockerfile`.
- **Container** — A running instance of an image (like an object).
- **Dockerfile** — A text file with instructions to build an image.
- **Registry** — A storage hub for images (e.g. Docker Hub).
- **Volume** — Persistent storage that survives container restarts.
- **Docker Compose** — A tool to define and run multi-container apps.

## 1. Checking Your Docker Installation

Run the cell below to verify Docker is installed and see the version.

In [2]:
import subprocess

# Works on both Windows and Mac
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
print("Docker Version:", result.stdout.strip())

result2 = subprocess.run(["docker", "info", "--format", "Server Version: {{.ServerVersion}}"], capture_output=True, text=True)
print(result2.stdout.strip() if result2.returncode == 0 else "Docker daemon not running — make sure Docker Desktop is open.")

Docker Version: Docker version 27.5.1, build 9f9e405801
Server Version: 29.1.5


## 2. Your First Container

The classic starting point — `hello-world` pulls a tiny image, runs it, prints a message, and exits.

```
docker run hello-world
```

**What happens step by step:**
1. Docker checks if the `hello-world` image exists locally
2. If not, it pulls it from Docker Hub
3. Docker creates a container from the image
4. The container runs, prints output, then stops

In [3]:
import subprocess

result = subprocess.run(["docker", "run", "hello-world"], capture_output=True, text=True)
print(result.stdout if result.stdout else result.stderr)


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/




## 3. Essential Docker Commands

These commands work identically on **Windows** (PowerShell / CMD) and **Mac** (Terminal).

| Command | What it does |
|---|---|
| `docker ps` | List **running** containers |
| `docker ps -a` | List **all** containers (including stopped) |
| `docker images` | List downloaded images |
| `docker pull <image>` | Download an image from Docker Hub |
| `docker run <image>` | Create and start a container |
| `docker run -d <image>` | Run container in background (detached) |
| `docker run -p 8080:80 <image>` | Map host port 8080 → container port 80 |
| `docker run -it <image> bash` | Open interactive terminal inside container |
| `docker stop <id>` | Gracefully stop a running container |
| `docker rm <id>` | Delete a stopped container |
| `docker rmi <image>` | Delete an image |
| `docker logs <id>` | View container output logs |
| `docker exec -it <id> bash` | Open a shell in a running container |

In [4]:
import subprocess

# List all local images
print("=== Local Images ===")
result = subprocess.run(["docker", "images"], capture_output=True, text=True)
print(result.stdout)

# List all containers (running + stopped)
print("=== All Containers ===")
result2 = subprocess.run(["docker", "ps", "-a"], capture_output=True, text=True)
print(result2.stdout)

=== Local Images ===
REPOSITORY                     TAG         IMAGE ID       CREATED        SIZE
redis                          latest      009cc37796fb   9 days ago     225MB
hello-world                    latest      452a468a4bf9   9 days ago     22.6kB
committee-node-dev             latest      e0618bde3702   2 weeks ago    1.56GB
committee-go-dev               latest      bda248729241   2 weeks ago    1.72GB
committee-mongo                latest      897dc3b00104   2 weeks ago    1.26GB
postgres                       16-alpine   20edbde7749f   5 weeks ago    389MB
mongo                          latest      3a7947f3211e   6 weeks ago    1.26GB
alfaarghya/alfa-leetcode-api   2.0.3       cba5e21dda08   3 months ago   643MB
minio/minio                    latest      14cea493d9a3   6 months ago   228MB
minio/mc                       latest      a7fe349ef4bd   6 months ago   112MB

=== All Containers ===
CONTAINER ID   IMAGE                                COMMAND                  CREAT

## 4. Writing a Dockerfile

A `Dockerfile` is a recipe for your image. Each line is an **instruction** (also called a layer).

| Instruction | Purpose |
|---|---|
| `FROM` | Base image to build on (always first) |
| `WORKDIR` | Set the working directory inside the container |
| `COPY` | Copy files from your machine into the image |
| `RUN` | Execute a command during the **build** step |
| `ENV` | Set environment variables |
| `EXPOSE` | Document which port the app listens on |
| `CMD` | Default command to run when the container starts |

The cell below creates a simple Python web app and its `Dockerfile` using Python's built-in file I/O — no shell differences between Windows and Mac.

In [5]:
import os

# Create a project folder (works on Windows and Mac)
project_dir = os.path.join(os.getcwd(), "my_docker_app")
os.makedirs(project_dir, exist_ok=True)

# --- app.py: a minimal HTTP server (no extra dependencies needed) ---
app_code = '''\
from http.server import HTTPServer, BaseHTTPRequestHandler

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-type", "text/html")
        self.end_headers()
        self.wfile.write(b"<h1>Hello from Docker!</h1><p>Running on Python inside a container.</p>")
    def log_message(self, format, *args):
        pass  # suppress access logs

if __name__ == "__main__":
    print("Server running on http://localhost:8080")
    HTTPServer(("", 8080), Handler).serve_forever()
'''

# --- Dockerfile ---
dockerfile = '''\
# Use the official lightweight Python image
FROM python:3.12-slim

# Set the working directory inside the container
WORKDIR /app

# Copy application source into the image
COPY app.py .

# Document that the container listens on port 8080
EXPOSE 8080

# Command to run when the container starts
CMD ["python", "app.py"]
'''

# Write files using Python (cross-platform)
with open(os.path.join(project_dir, "app.py"), "w") as f:
    f.write(app_code)

with open(os.path.join(project_dir, "Dockerfile"), "w") as f:
    f.write(dockerfile)

print(f"Created project at: {project_dir}")
print("\n--- Dockerfile contents ---")
print(dockerfile)

Created project at: /Users/henriquepitta/Desktop/Sprinternship-Code/my_docker_app

--- Dockerfile contents ---
# Use the official lightweight Python image
FROM python:3.12-slim

# Set the working directory inside the container
WORKDIR /app

# Copy application source into the image
COPY app.py .

# Document that the container listens on port 8080
EXPOSE 8080

# Command to run when the container starts
CMD ["python", "app.py"]



## 5. Building and Running Your Image

**Build** converts your `Dockerfile` into an image stored locally.  
**Run** starts a container from that image.

```
# Build the image and tag it "my-app"
docker build -t my-app ./my_docker_app

# Run it — map host port 8080 to container port 8080
# -d runs it in the background (detached mode)
docker run -d -p 8080:8080 --name my-app-container my-app
```

After running, open your browser to **http://localhost:8080** on either OS.

In [6]:
import subprocess, os

project_dir = os.path.join(os.getcwd(), "my_docker_app")

# Build the image
print("Building image... (this may take a moment on first run)")
build = subprocess.run(
    ["docker", "build", "-t", "my-app", project_dir],
    capture_output=True, text=True
)
print(build.stdout[-500:] if build.stdout else "")  # print last 500 chars to keep output tidy
if build.returncode != 0:
    print("Build error:", build.stderr[-300:])
else:
    print("Image built successfully!\n")

    # Stop and remove any previous run of this container (so re-running this cell works)
    subprocess.run(["docker", "stop", "my-app-container"], capture_output=True)
    subprocess.run(["docker", "rm",   "my-app-container"], capture_output=True)

    # Start the container detached on port 8080
    run = subprocess.run(
        ["docker", "run", "-d", "-p", "8080:8080", "--name", "my-app-container", "my-app"],
        capture_output=True, text=True
    )
    if run.returncode == 0:
        print("Container started! Visit http://localhost:8080 in your browser.")
        print("Container ID:", run.stdout.strip()[:12])
    else:
        print("Run error:", run.stderr)

Building image... (this may take a moment on first run)

Image built successfully!

Container started! Visit http://localhost:8080 in your browser.
Container ID: bb86d834f77e


## 6. Docker Compose

**Docker Compose** lets you define a multi-container application in a single `docker-compose.yml` file instead of running long `docker run` commands.

Key commands (same on Windows and Mac):

| Command | What it does |
|---|---|
| `docker compose up` | Build (if needed) and start all services |
| `docker compose up -d` | Start in background |
| `docker compose down` | Stop and remove containers |
| `docker compose logs` | View logs from all services |
| `docker compose ps` | List running services |

The example below defines **two services**: our Python app + a Redis cache — a common real-world pattern.

In [7]:
import os

project_dir = os.path.join(os.getcwd(), "my_docker_app")

compose_yaml = '''\
version: "3.9"

services:
  web:
    build: .                  # Build from the Dockerfile in this folder
    ports:
      - "8080:8080"           # host:container port mapping
    environment:
      - REDIS_HOST=redis      # service name acts as hostname inside Docker network
    depends_on:
      - redis                 # ensure redis starts first

  redis:
    image: redis:7-alpine     # pull official Redis image (no Dockerfile needed)
    ports:
      - "6379:6379"
'''

with open(os.path.join(project_dir, "docker-compose.yml"), "w") as f:
    f.write(compose_yaml)

print("Created docker-compose.yml")
print()
print(compose_yaml)
print("To start both services, run from the my_docker_app folder:")
print("  docker compose up -d")
print()
print("To stop everything:")
print("  docker compose down")

Created docker-compose.yml

version: "3.9"

services:
  web:
    build: .                  # Build from the Dockerfile in this folder
    ports:
      - "8080:8080"           # host:container port mapping
    environment:
      - REDIS_HOST=redis      # service name acts as hostname inside Docker network
    depends_on:
      - redis                 # ensure redis starts first

  redis:
    image: redis:7-alpine     # pull official Redis image (no Dockerfile needed)
    ports:
      - "6379:6379"

To start both services, run from the my_docker_app folder:
  docker compose up -d

To stop everything:
  docker compose down


## 7. Cleaning Up

Containers and images accumulate quickly. Run the cell below to stop and remove what we created in this tutorial.

## 8. Best Practices & Next Steps

### Best Practices

**Dockerfile**
- Always pin your base image version: `FROM python:3.12-slim` not `FROM python:latest`
- Use `.dockerignore` (like `.gitignore`) to exclude `node_modules`, `.git`, etc. from builds
- Order layers from least → most frequently changed to maximize build cache
- Combine `RUN` commands with `&&` to reduce layers
- Run as a non-root user for security: add `RUN useradd -m appuser && USER appuser`

**Images**
- Prefer `-slim` or `-alpine` variants to keep image sizes small
- Scan images for vulnerabilities: `docker scout cves <image>`

**Containers**
- Use `-e` flags or `.env` files for secrets — never bake them into the image
- Use named volumes for persistent data (`docker volume create mydata`)
- Always set resource limits in production (`--memory`, `--cpus`)

---

### Next Steps

| Topic | Command / Resource |
|---|---|
| Push image to Docker Hub | `docker push yourusername/my-app` |
| Multi-stage builds | Reduce image size by separating build & runtime stages |
| Docker volumes | `docker volume create` / `docker run -v` |
| Networking | `docker network create` — connect containers |
| Kubernetes | The next step after Docker for orchestration at scale |
| Docker Desktop Dashboard | GUI alternative for all commands above |